<a href="https://colab.research.google.com/github/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/blob/main/Model_Selection_Notebook_2nd_edition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Selection ⚙️



## Establishing Data Assumptions

Our Exploratory Data Analysis (EDA) and statistical testing revealed critical structural realities about the film industry's financial data. Specifically, we observed:

* Variance Violation: We have severe heteroscedasticity in the relationship between capital effort and revenue.
* Non-Normality Violation: Our target variable distribution is heavily right skewed, fat-tailed.

Because standard parametric models rely on strict assumptions of linearity and constant variance, relying solely on them would yield unstable predictions.

Therefore, **our core assumption is that the interactions driving movie profitability** (e.g., the synergy of a top director, high budget, and specific genre) **are complex and non-linear**.

## Algorithm Selection

To accommodate these assumptions and capture complex interactions, we will utilize a progressive modeling architecture based on statistical machine learning principles:

* **Regularized Logistic Regression (L1/Lasso):** We will deploy this as our linear, interpretable baseline model. By using an $L1$ penalty, we force the algorithm to perform automatic feature selection, shrinking the coefficients of irrelevant predictors exactly to zero.

* **Random Forest:** A bagging ensemble method that trains numerous independent decision trees on bootstrapped data subsets. It naturally handles non-linearities and does not require scaled data, effectively bypassing our heteroscedasticity concerns while remaining highly robust against overfitting.

* **XGBoost (Extreme Gradient Boosting):** Boosting algorithm builds trees sequentially, with each new tree correcting the residual errors of its predecessors. It is highly optimized for extracting patterns from complex, high-dimensional data, similar to our entertainment dataset.

### Setup, Data Leakage Prevention, and Splitting

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

!wget https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/greenlight_model_data.csv -O greenlight_model_data.csv

# 1. Load the data
model_df = pd.read_csv('greenlight_model_data.csv')

# 2. Prevent Data Leakage
leakage = model_df[['real_profit','real_revenue','log_real_revenue','ROI','score','vote_count',
                    'director_win_rate','star_win_rate','writer_win_rate','production_win_rate','rating_win_rate']]
y = model_df['is_profitable']
leakage_and_target = leakage.columns.tolist() + [y.name]
X = model_df.drop(columns=leakage_and_target, errors='ignore')

# One-hot encode remaining categorical features
X = pd.get_dummies(X, drop_first=True)

# FIX FOR XGBOOST: Strip forbidden characters ([, ], <, >) from column names
X.columns = X.columns.str.replace(r'[\[\]<,]', '', regex=True)

# 3. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Feature Scaling (Required for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize an empty list to hold the outside evaluation metrics
results = []
print("Data successfully loaded, secured against leakage, and cleaned for XGBoost.")

--2026-03-17 23:42:45--  https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/greenlight_model_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1832495 (1.7M) [text/plain]
Saving to: ‘greenlight_model_data.csv’

greenlight_model_da 100%[===================>]   1.75M  --.-KB/s    in 0.09s   

2026-03-17 23:42:46 (18.7 MB/s) - ‘greenlight_model_data.csv’ saved [1832495/1832495]

Data successfully loaded, secured against leakage, and cleaned for XGBoost.


### Logistic Regression (L1 Baseline)

In [24]:
# Initialize Model
lr_model = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=1000)

# Inside Evaluation: 5-Fold Cross-Validation
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='f1')
print(f"Logistic Regression CV F1-Score: {lr_cv_scores.mean():.4f} (Std = {lr_cv_scores.std():.4f})")

# Outside Evaluation: Train and Predict on Holdout
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# Append metrics to results list
results.append({
    'Model': 'Logistic Regression (L1)',
    'Accuracy': accuracy_score(y_test, lr_pred),
    'Precision': precision_score(y_test, lr_pred),
    'Recall': recall_score(y_test, lr_pred),
    'F1-Score': f1_score(y_test, lr_pred),
    'AUC-ROC': roc_auc_score(y_test, lr_prob)
})
print("Logistic Regression successfully evaluated.")

Logistic Regression CV F1-Score: 0.8012 (Std = 0.0011)
Logistic Regression successfully evaluated.


### Random Forest

In [25]:
# Initialize Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)

# Inside Evaluation: 5-Fold Cross-Validation (Using unscaled data as trees don't require scaling)
rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='f1')
print(f"Random Forest CV F1-Score: {rf_cv_scores.mean():.4f} (Std = {rf_cv_scores.std():.4f})")

# Outside Evaluation: Train and Predict on Holdout
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Append metrics to results list
results.append({
    'Model': 'Random Forest',
    'Accuracy': accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred),
    'Recall': recall_score(y_test, rf_pred),
    'F1-Score': f1_score(y_test, rf_pred),
    'AUC-ROC': roc_auc_score(y_test, rf_prob)
})
print("Random Forest successfully evaluated.")

Random Forest CV F1-Score: 0.8072 (Std = 0.0003)
Random Forest successfully evaluated.


### XGBoost

In [26]:
# Initialize Model
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, max_depth=6)

# Inside Evaluation: 5-Fold Cross-Validation (Using unscaled data)
xgb_cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='f1')
print(f"XGBoost CV F1-Score: {xgb_cv_scores.mean():.4f} (Std = {xgb_cv_scores.std():.4f})")

# Outside Evaluation: Train and Predict on Holdout
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# Append metrics to results list
results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, xgb_pred),
    'Precision': precision_score(y_test, xgb_pred),
    'Recall': recall_score(y_test, xgb_pred),
    'F1-Score': f1_score(y_test, xgb_pred),
    'AUC-ROC': roc_auc_score(y_test, xgb_prob)
})
print("XGBoost successfully evaluated.")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:44:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:45:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:45:41] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:46:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

XGBoost CV F1-Score: 0.7865 (Std = 0.0103)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:47:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost successfully evaluated.


# Model Evaluation

## Evaluation Strategy

To ensure our models generalize effectively to future, unproduced films, we will employ a dual-layered evaluation methodology:

* **Inside Evaluation (Cross-Validation):** We will use $K$-Fold Cross Validation ($k=5$) on the training set to ensure model stability across different data subsets.

* **Outside Evaluation (Holdout Metrics):** We will evaluate the models on a 20% unseen test set. Because our dataset has a slight class imbalance (a 67.7% baseline hit rate), relying solely on Accuracy is dangerous. We will evaluate performance using Precision, Recall, the F1-Score, and the Area Under the Receiver Operating Characteristic Curve (AUC-ROC).

In [27]:
# Display the consolidated metrics for all models
results_df = pd.DataFrame(results)
print("\n--- Outside Evaluation: Holdout Test Set Metrics ---")
print(results_df.to_string(index=False))


--- Outside Evaluation: Holdout Test Set Metrics ---
                   Model  Accuracy  Precision   Recall  F1-Score  AUC-ROC
Logistic Regression (L1)  0.668856   0.690402 0.926593  0.791248 0.658929
           Random Forest  0.677298   0.677298 1.000000  0.807606 0.665253
                 XGBoost  0.687617   0.732378 0.849030  0.786402 0.696203


## 🧠 Step 4: Interpreting the Results (Business & Data Narrative)
Our models have been evaluated on the unseen holdout data, and the metrics provide a fascinating look into the predictability of the movie business. Rather than just looking at raw accuracy, we must interpret these algorithms through the lens of a studio executive deciding whether to risk millions of dollars.

1. **The "Play it Safe" Trap (Random Forest):**
At first glance, the Random Forest ensemble appears to have the highest F1-Score (0.807). However, looking closely at its Recall (1.000) and Precision (0.677), we uncover a classic machine learning pitfall. A Precision of 0.677 perfectly matches our dataset's baseline hit rate (67.7%). This means the Random Forest simply predicted that every single movie in the test set would be profitable. The algorithm realized that the easiest way to minimize overall error in a favorable market was to greenlight everything. While technically accurate 67% of the time, a model that never predicts a box office bomb provides zero strategic value to a studio.

2. **The Linear Baseline (Logistic Regression):**
The L1-penalized Logistic Regression model attempted to find strict linear rules for success. While it performed slightly better at actually separating the classes than the Random Forest, it still heavily over-predicted success (Recall of 0.926).

      This mathematically confirms our Phase 2 assumption: the rules of Hollywood profitability are highly non-linear. Simple, straight-line equations cannot adequately capture the complex synergy of talent, budget, and release timing.

3. **The Clear Winner: XGBoost:**
XGBoost, our most complex algorithm, emerges as the definitive winner, achieving the highest Accuracy (0.687), the highest Precision (0.732), and the highest AUC-ROC (0.696).

    * <u>The Business Value</u>: Unlike the other models, XGBoost actually learned to identify financial failures. By sacrificing some Recall (0.849), it achieved a Precision of 0.732. In business terms, this means when XGBoost recommends greenlighting a project, the probability of it actually being profitable is noticeably higher than the industry baseline.

    * <u>The AUC-ROC Signal</u>: The AUC score (0.696) measures the model's ability to rank a successful movie higher than a flop. A score near 0.70 indicates a solid, actionable predictive signal. It proves that pre-release metrics do contain the DNA of a movie's financial destiny, successfully reducing uncertainty despite the inherent chaos of consumer entertainment.

### **Bridging the "Black Box" Interpretability Gap:**

By selecting XGBoost as our primary predictive engine, we gain accuracy but sacrifice the simple transparency of Logistic Regression. To ensure stakeholders and executives trust these predictions, our next operational step must involve model explainability. Rather than showing executives complex decision trees, we will deploy interpretability frameworks (such as SHAP values) to break down individual predictions. This allows us to definitively explain why the algorithm made its choice (e.g., "The model predicts a 75% chance of profit, primarily driven by the Director's high historical win rate and the optimal holiday release window, despite the high production budget").